# Tema 5 · Laboratorio — Generar dígitos con una GAN

**Aprendizaje Profundo · CUNEF Universidad**

La panorámica del Tema 5 presentaba los modelos **generativos**. Aquí construimos una **GAN** (Generative Adversarial Network) mínima que aprende a fabricar dígitos MNIST a partir de **ruido**.

Dos redes compiten:
- El **generador** transforma ruido en imágenes falsas y quiere **engañar**.
- El **discriminador** mira una imagen y decide si es **real o falsa**.

Entrenar la GAN es ese duelo: cada uno mejora empujado por el otro.

> **Usa GPU** (Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*). Ejecuta las celdas en orden.

## 1 · Datos: MNIST escalado a [-1, 1]

El generador terminará en una `tanh` (salida en [-1, 1]), así que escalamos las imágenes reales al mismo rango.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.random.set_seed(42)

(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype('float32') - 127.5) / 127.5   # -> [-1, 1]
x_train = x_train[..., None]                            # (N, 28, 28, 1)

BATCH = 128
LATENT = 100
dataset = (tf.data.Dataset.from_tensor_slices(x_train)
           .shuffle(60000).batch(BATCH, drop_remainder=True).prefetch(tf.data.AUTOTUNE))
print('imágenes de entrenamiento:', x_train.shape)

## 2 · El generador

Recibe un vector de ruido de dimensión 100 y lo va **ampliando** (con `Conv2DTranspose`) hasta una imagen de 28×28. La última capa usa `tanh`.

In [ ]:
def make_generator():
    return keras.Sequential([
        keras.layers.Input(shape=(LATENT,)),
        layers.Dense(7 * 7 * 128, use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(64, 4, strides=2, padding='same', use_bias=False),  # 14x14
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(1, 4, strides=2, padding='same', activation='tanh'),  # 28x28
    ], name='generador')

generator = make_generator()
generator.summary()

## 3 · El discriminador

Una CNN clasificadora (Tema 4): recibe una imagen 28×28 y produce un único número (logit) — cuanto mayor, más 'real' la cree.

In [ ]:
def make_discriminator():
    return keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(64, 4, strides=2, padding='same'),   # 14x14
        layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Conv2D(128, 4, strides=2, padding='same'),  # 7x7
        layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1),   # logit (sin activación)
    ], name='discriminador')

discriminator = make_discriminator()
discriminator.summary()

## 4 · Pérdidas y el paso de entrenamiento

Usamos entropía cruzada binaria sobre los logits. En cada paso:
1. El **discriminador** ve un lote real (etiqueta 1) y uno falso (etiqueta 0) y aprende a separarlos.
2. El **generador** se actualiza para que el discriminador clasifique sus falsificaciones como reales.

In [ ]:
bce = keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = keras.optimizers.Adam(1e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5)

@tf.function
def train_step(real_images):
    noise = tf.random.normal([BATCH, LATENT])
    with tf.GradientTape() as g_tape, tf.GradientTape() as d_tape:
        fake_images = generator(noise, training=True)
        real_out = discriminator(real_images, training=True)
        fake_out = discriminator(fake_images, training=True)
        # discriminador: reales -> 1, falsas -> 0
        d_loss = bce(tf.ones_like(real_out), real_out) + bce(tf.zeros_like(fake_out), fake_out)
        # generador: quiere que las falsas se clasifiquen como reales -> 1
        g_loss = bce(tf.ones_like(fake_out), fake_out)
    g_opt.apply_gradients(zip(g_tape.gradient(g_loss, generator.trainable_variables),
                              generator.trainable_variables))
    d_opt.apply_gradients(zip(d_tape.gradient(d_loss, discriminator.trainable_variables),
                              discriminator.trainable_variables))
    return g_loss, d_loss

## 5 · Entrenar

Con ~30 épocas empiezan a verse dígitos reconocibles. Cuantas más entrenes (y con GPU), mejor. Guardamos las pérdidas para verlas después.

In [ ]:
EPOCHS = 30
semilla_fija = tf.random.normal([16, LATENT])  # el mismo ruido en cada muestra, para ver la evolución
g_hist, d_hist = [], []

def muestra(epoch):
    imgs = generator(semilla_fija, training=False)
    imgs = (imgs + 1) / 2   # a [0, 1] para visualizar
    plt.figure(figsize=(4, 4))
    for i in range(16):
        plt.subplot(4, 4, i+1); plt.imshow(imgs[i, :, :, 0], cmap='gray'); plt.axis('off')
    plt.suptitle(f'época {epoch}'); plt.tight_layout(); plt.show()

for epoch in range(1, EPOCHS + 1):
    gl, dl = [], []
    for real in dataset:
        g, d = train_step(real)
        gl.append(float(g)); dl.append(float(d))
    g_hist.append(np.mean(gl)); d_hist.append(np.mean(dl))
    if epoch % 10 == 0 or epoch == 1:
        print(f'época {epoch:>3}  g_loss={g_hist[-1]:.3f}  d_loss={d_hist[-1]:.3f}')
        muestra(epoch)

## 6 · Las dos pérdidas

En una GAN **no** buscamos que la pérdida baje a cero: buscamos un **equilibrio** entre generador y discriminador. Si una domina a la otra, el entrenamiento se rompe.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(g_hist, label='generador')
plt.plot(d_hist, label='discriminador')
plt.title('Pérdidas de la GAN (buscamos equilibrio, no cero)')
plt.xlabel('época'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 7 · Tus retos

1. **Más épocas.** Sube `EPOCHS` a 60–100 (con GPU). ¿Mejoran los dígitos?
2. **El ruido latente.** Genera dígitos con dos vectores de ruido y muestra la **interpolación** entre ellos (mezcla `z = (1-t)·z1 + t·z2`). ¿Se transforma un dígito en otro suavemente?
3. **Equilibrio.** Entrena el discriminador **dos veces** por cada paso del generador. ¿Cambia la estabilidad?
4. **VAE (avanzado).** Compara con un *autoencoder variacional*: otra familia generativa que optimiza una pérdida distinta y suele ser más estable.

Recuerda el paralelismo con la práctica de T5: el **ataque adversario** también enfrenta dos objetivos; aquí esa idea adversaria es el motor que *genera*.